# **Lecture 19: Solving PDEs I - The Heat/Diffusion Equation**
**Date:** Tuesday, 17-02-2026

**Unit:** 2 (Differential Equations)

**Topic:** Solving PDEs I: Heat/Diffusion Equation using numpy meshgrids

---

## **1. The Physics of Diffusion**

In Lecture 17, we classified PDEs. Today, we solve our first **Parabolic PDE**: The Diffusion Equation (often called the Heat Equation).

In Astrophysics, this equation does not just describe thermal heat. It governs anything that "random walks" or scatters through a medium to smooth out concentration gradients:
* **Radiative Transfer:** Photons diffusing from the core of the Sun to the surface (which takes ~100,000 years!).
* **Chemical Mixing:** Heavy elements diffusing through a stellar envelope.
* **Thermal Conduction:** Heat moving outward through a white dwarf.

The 1D Heat Equation is:
$$\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2}$$

Where:
* $u(x, t)$ is the field (e.g., Temperature).
* $\alpha$ is the thermal diffusivity constant (how fast the medium allows heat to spread).

## **2. The FTCS Scheme (Forward Time, Centered Space)**

To solve this numerically, we use the Finite Difference Method (FDM) grid we built in Lecture 17.

1.  **Time Derivative (LHS):** We use a **Forward Difference** to "march" into the future.
    $$\frac{\partial u}{\partial t} \approx \frac{u_i^{n+1} - u_i^n}{\Delta t}$$
2.  **Space Derivative (RHS):** We use a **Central Difference** to measure the spatial curvature of the heat.
    $$\frac{\partial^2 u}{\partial x^2} \approx \frac{u_{i+1}^n - 2u_i^n + u_{i-1}^n}{\Delta x^2}$$

Equating these and solving for the future state $u_i^{n+1}$, we get the explicit **FTCS Update Equation**:
$$u_i^{n+1} = u_i^n + \frac{\alpha \Delta t}{\Delta x^2} \left( u_{i+1}^n - 2u_i^n + u_{i-1}^n \right)$$

**Physical Intuition:** The future temperature is the current temperature PLUS a small adjustment. If the surrounding points ($u_{i+1}$ and $u_{i-1}$) are hotter than the center point ($2u_i$), the curvature is positive, and heat flows IN. If the center is hotter, heat flows OUT.

## **3. Python Implementation (1D)**

Let's simulate a 1D rod (or a core-to-surface radial slice of a star). We will use vectorization (array slicing) to make the code fast.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Setup the Grid
L = 100.0          # Length of domain
nx = 100           # Number of spatial grid points
dx = L / (nx - 1)
x = np.linspace(0, L, nx)

# 2. Physical Parameters
alpha = 2.0        # Diffusivity
dt = 0.1           # Time step size
t_total = 50.0     # Total simulation time
n_steps = int(t_total / dt)

# 3. Initial Conditions: A sharp "Hot Spot" in the center
u = np.zeros(nx)
u[40:60] = 100.0   # 100 degrees in the middle, 0 elsewhere

# Save initial state for plotting later
u_initial = u.copy()

# 4. The FTCS Time Loop (Vectorized)
for n in range(n_steps):
    # Calculate the second derivative using array slicing (interior points only)
    d2u_dx2 = (u[2:] - 2*u[1:-1] + u[:-2]) / dx**2
    
    # Update the interior points
    u[1:-1] = u[1:-1] + alpha * dt * d2u_dx2
    
    # Boundary Conditions (Dirichlet: Ends are kept at 0 degrees)
    u[0] = 0.0
    u[-1] = 0.0

# 5. Visualization
plt.figure(figsize=(10, 5))
plt.plot(x, u_initial, 'k--', label='Initial Temperature (t=0)')
plt.plot(x, u, 'r-', lw=2, label=f'Final Temperature (t={t_total})')
plt.title('1D Heat Diffusion over Time')
plt.xlabel('Position x')
plt.ylabel('Temperature')
plt.legend()
plt.grid(True)
plt.show()


## **4. Stepping into 2D with `numpy.meshgrid`**

Astrophysics is rarely 1D. To simulate a 2D cross-section of a nebula or an accretion disk, we need the 2D Heat Equation:
$$\frac{\partial u}{\partial t} = \alpha \left( \frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2} \right)$$

To do this efficiently in Python, we use `numpy.meshgrid`. This function takes 1D coordinate arrays and broadcasts them into 2D coordinate matrices, allowing us to evaluate functions across an entire 2D plane without writing nested `for` loops.

In [ ]:
# 1. Setup the 2D Spatial Grid
Lx, Ly = 50.0, 50.0
nx, ny = 60, 60
x = np.linspace(-Lx/2, Lx/2, nx)
y = np.linspace(-Ly/2, Ly/2, ny)
dx = x[1] - x[0]
dy = y[1] - y[0]

# Generate the 2D Meshgrid
X, Y = np.meshgrid(x, y)

# 2. Initial Condition: A Gaussian hot spot offset from the center
u_2d = 100.0 * np.exp(-0.1 * ((X - 10)**2 + (Y + 10)**2))

# 3. Physical Parameters
alpha_2d = 1.5
dt_2d = 0.05  # Must be small for stability!
steps_2d = 400

# 4. The 2D FTCS Loop (Vectorized)
for n in range(steps_2d):
    # X-direction curvature
    u_xx = (u_2d[1:-1, 2:] - 2*u_2d[1:-1, 1:-1] + u_2d[1:-1, :-2]) / dx**2
    
    # Y-direction curvature
    u_yy = (u_2d[2:, 1:-1] - 2*u_2d[1:-1, 1:-1] + u_2d[:-2, 1:-1]) / dy**2
    
    # Update interior points
    u_2d[1:-1, 1:-1] = u_2d[1:-1, 1:-1] + alpha_2d * dt_2d * (u_xx + u_yy)
    
    # (Edges naturally remain at their initial 0.0 values, acting as fixed boundaries)

# 5. Visualization
plt.figure(figsize=(8, 6))
# contourf creates filled contour plots, perfect for 2D fields
cp = plt.contourf(X, Y, u_2d, levels=50, cmap='inferno')
plt.colorbar(cp, label='Temperature')
plt.title('2D Heat Diffusion (After 400 steps)')
plt.xlabel('X position')
plt.ylabel('Y position')
plt.show()


## **5. Student Exercises**

### **Exercise 1: Neumann Boundary Conditions (Insulation)**
In the 1D example above, we forced the edges of the rod to stay at exactly $0$ degrees (Dirichlet boundary condition). This acts like an infinite ice bath.

What if the rod is wrapped in perfect insulation? This means heat cannot flow out the ends. Mathematically, the derivative at the boundary must be zero: $\frac{\partial u}{\partial x} = 0$.
In our discrete grid, this means $u_0 = u_1$ and $u_{N-1} = u_{N-2}$.

**Task:** Copy the 1D code block from Section 3. Modify the boundary condition lines inside the `for` loop to implement perfect insulation. Plot the result. What happens to the total heat in the rod compared to the original code?

---

### **Exercise 2: The Stability Limit (Foreshadowing Lecture 20)**
In explicit FDM schemes like FTCS, you cannot make your time step $\Delta t$ as large as you want. If the simulation steps too far forward in time before checking the spatial curvature, the math "overshoots" and explodes to infinity.

For the 1D Heat Equation, the stability requirement is:
$$\Delta t \le \frac{\Delta x^2}{2\alpha}$$

**Task:** 
1. In the 1D code from Section 3, calculate the maximum allowed $\Delta t$ based on this formula.
2. Change `dt` in the code to a value slightly *larger* than this limit and run the code. 
3. Observe the numerical artifact (checkerboarding) that destroys the simulation. This is why implicit methods or adaptive time-stepping are required in real astrophysics codes.

In [ ]:
# Student Code Area
import numpy as np
import matplotlib.pyplot as plt

# Implement Exercise 1 or 2 here...
